## Newtons Gravitationsgesetz
Die Simulation soll mit Newtons Gravitationsgesetz implementiert werden:

$$
\vec{F}_{ji} = -G \cdot \frac{m_i \cdot m_j}{\|\vec{r}_{ji}\|^2}
$$

- $F_{ji}$ ist dabei die Kraft, die ein Körper $j$ auf einen Körper $i$ auswirkt.
- $m_i, m_j$ sind die Massen der beiden Körper und
- $\vec{r}_{ji}$ ist deren Abstand.
- $G$ ist die Gravitationskonstante mit $G\approx 6,67430\cdot 10^{-11}\frac{m^3}{kg\cdot s^2}$.
- Die wirkende Kraft ist negativ, da es sich um eine anziehende Kraft handelt.
- $\vec{F}_{ji} = -\vec{F}_{ij}$ da beide Körper eine betragsmäßig gleichgroße Kraft in entgegengesetzte Richtung aufeinander wirken (Wechselwirkungsgesetz).

Mit diesem Gravitationsgesetz soll nun die Simulation implementiert werden.

In [ ]:
import numpy as np
import math

class Body:
    
    def __init__(self, name:str, position, velocity, radius:float, mass:float, color: str):
        self.name = name
        self.position = np.array(position, dtype=float) # Typkonvertierung um sicherzustellen, dass es immer NumPy-Arrays sind
        self.velocity = np.array(velocity, dtype=float)
        self.radius = radius
        self.diameter = 2*radius
        self.mass = mass
        self.color = color # Farbe des Körpers für die spätere Visualisierung
        
    def get_distance_to_body(self, other: 'Body') -> float:
        
        ab = other.position - self.position
        distance = np.linalg.norm(ab)
        
        return distance
    
    
    def update_position(self, timeStep: float):
        self.position = self.position + self.velocity * timeStep
        
    def update_velocity(self, acceleration: np.ndarray, timeStep: float):
        self.velocity = self.velocity + acceleration * timeStep
        
    def get_momentum(self)-> np.ndarray:
        return self.mass*self.velocity

In [ ]:
class Simulation:
    def __init__(self, time_step):
        self.bodies = []
        self.G = 6.67430e-11 * (86400 ** 2)  # Umrechnung von m³/(kg·s²) auf m³/(kg·Tag²)
        self.time_step = time_step
        self.history = {}

    def add_body(self, body):
        if any(b.name == body.name for b in self.bodies):
            raise ValueError(f"Ein Himmelskörper mit dem Namen '{body.name}' existiert bereits!")
        self.bodies.append(body)
        self.history[body.name] = []

    def remove_body(self, body):
        if body in self.bodies:
            self.bodies.remove(body)

    def calculate_gravity(self):
        # Dictionary mit allen Körpern
        forces = {body.name : np.zeros(3) for body in self.bodies}

        for i in range(len(self.bodies)):
            for j in range(i + 1, len(self.bodies)):
                b1 = self.bodies[i]
                b2 = self.bodies[j]
                
                # Abstandsvektor von b2 zu b1
                r_vector = b1.position - b2.position
                distance = np.linalg.norm(r_vector)
                
                if distance == 0:
                    continue # zur Sicherheit gegen Division durch Null
                
                # Berechne den Betrag der Kraft nach Newton
                force_magnitude = self.G * b1.mass * b2.mass / (distance**2)
                
                # Richtungsvektor (Einheitsvektor)
                direction = r_vector / distance
                
                # Gesamtkraftvektor
                force_vector = force_magnitude * direction
                
                # Wechselwirkungsgesetz
                forces[b2.name] += force_vector  # b2 wird von b1 angezogen
                forces[b1.name] -= force_vector  # b1 wird von b2 angezogen
                
        return forces

    def check_collisions(self):
        collCheck = False
        found_collision = True
        while found_collision:
            found_collision = False
            for i in range(len(self.bodies)):
                for j in range(i+1, len(self.bodies)):
                    b1 = self.bodies[i]
                    b2 = self.bodies[j]

                    # Abstand zwischen den zwei zu prüfenden Körpern
                    distance = np.linalg.norm(b1.position - b2.position)

                    if (distance <= (b1.radius + b2.radius)):
                        self.merge_bodies(b1, b2)
                        collCheck = True
                        found_collision = True
                        break # Liste hat sich geändert, Suche neu starten
                if found_collision:
                    break
        # wir sind die ganze Liste durchgegangen und haben alle Kollisionen entdeckt
        return collCheck

    def merge_bodies(self, body1, body2):

        # neue Attribute für den verschmelzten Body
        newMass = body1.mass + body2.mass
        newRadius = math.sqrt((body1.radius**2) + (body2.radius**2))
        newVeloc = ((body1.mass/newMass) * body1.velocity) + ((body2.mass/newMass) * body2.velocity)
        newPos = (body1.mass * body1.position + body2.mass * body2.position) / newMass # Schwerpunkt: Division durch Gesamtmasse
        
        # die beiden vorherigen Bodies werden entfernt
        self.bodies.remove(body1)
        self.bodies.remove(body2)

        # neuer Body
        newBody = Body(body1.name + "+" + body2.name, newPos, newVeloc, newRadius, newMass, "")
        self.add_body(newBody)
        

    def step(self):
        self.check_collisions()
        
        # Kräfte berechnen
        forces = self.calculate_gravity()
        
        for body in self.bodies:
            if body.name in forces: # zur Sicherheit gegen Fehler
                # F = m*a umformen zu a = F/m, da update_velocity eine Beschleunigung erwartet
                acceleration = forces[body.name] / body.mass
                body.update_velocity(acceleration, self.time_step)
                body.update_position(self.time_step)
    
                # Verlauf speichern für die spätere Visualisierung
                self.history[body.name].append(body.position.copy())

    def run(self, total_time):
        # Berechnet, wie viele Schritte nötig sind
        steps = int(total_time / self.time_step)
        for _ in range(steps):
            self.step()

Die Klasse Simulation implementiert die Physik des Programms. Sie agiert als diskrete Physik-Engine, welche die zeitliche Evolution des Mehrkörper-Systems (N-Körper-Problem) auf Basis der klassischen Newtonschen Mechanik berechnet.  

Die Klasse erfüllt dabei folgende Kernaufgaben:
- Verwaltung des Systems: Sie registriert alle am System beteiligten Himmelskörper (Body-Objekte) und initialisiert Datenstrukturen für die Speicherung der Trajektorien.

- Numerische Integration: Da eine analytische Lösung für N>2 Körper nicht existiert, nutzt die Engine ein numerisches Integrationsverfahren. Sie berechnet in jedem konfigurierbaren Zeitschritt ($\Delta t$) die aktuellen Beschleunigungen, Geschwindigkeiten und Positionen aller Körper.

- Gravitative Wechselwirkung: Über die Methode calculate_gravity wird paarweise der Gravitationsvektor nach Newtons Gravitationsgesetz ermittelt.  

- Kollisionsmanagement: In jedem Simulationsschritt wird geprüft, ob sich die physikalischen Radien zweier Körper überschneiden. Bei Kontakt wird eine 100% inelastische Kollision simuliert, bei der die Massen und Impulse addiert werden und ein neuer, volumenbasierter Radius berechnet wird.

- Historisierung: Für die anschließende Visualisierung trackt die Klasse alle vergangenen Positionen (history), sodass Flugbahnen und Animationen exakt rekonstruiert werden können.

## Visualisierung

In [ ]:

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

class Visualizer:
    def __init__(self, simulation, distance_scale=1.0, body_scale=1.0):
        """
        Initialisiert den Visualizer für die 3D Körper-Simulation.
        
        Parameters:
        - simulation: Das fertige Simulations-Objekt mit den berechneten Daten.
        - distance_scale: Faktor, mit dem alle Abstände multipliziert werden.
        - body_scale: Faktor, mit dem die Radien der Körper vergrößert werden.
        """
        self.simulation = simulation
        self.distance_scale = distance_scale
        self.body_scale = body_scale
    
    def animate_3d(self, view='standard'):
        """
        Erstellt eine dynamische 3D-Animation aus den echten Simulationsdaten.
        """
        # Stehen uns Simulationsdaten zur Verfügung?
        if self.simulation is None or not hasattr(self.simulation, 'history'):
            raise ValueError("Keine Simulationsdaten gefunden! Bitte sicherstellen, dass die Simulation lief und 'history' existiert.")
            
        # Greife auf Simulationsdaten zu
        data_source = self.simulation.history
        
        # Initialisiere 3D-Plot
        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='3d') 

        # Kameraperspektive anpassen, je nach Wunsch
        if view == 'top':
            # Schaut direkt von oben auf die X/Y-Ebene hinab (wie 2D)
            ax.view_init(elev=90, azim=-90)
            ax.set_title('Simulation (Draufsicht)')
        elif view == 'side':
            # Schaut flach von der Seite auf die X/Z-Ebene
            ax.view_init(elev=0, azim=-90)
            ax.set_title('Simulation (Seitenansicht)')
        else:
            # Standard-Perspektive (schräg von oben)
            ax.view_init(elev=30, azim=-60)
            ax.set_title('Simulation (Standardansicht)')
        
        # Dictionaries werden genutzt um die Daten für jeden Körper einzeln zu speichern
        lines = {}   # Für die Flugbahnen (Trajektorien)
        points = {}  # Für die aktuellen Positionen der Planeten
        
        # Jeder Körper wird einzeln mit seiner individuellen Farbe geplotet
        for body in self.simulation.bodies:
            name = body.name
            color = getattr(body, 'color', 'blue')  # Fallback auf Blau, falls mal keine Farbe definiert ist
            
            # Die Flugbahn bekommt die Farbe (gerne etwas blasser via alpha)
            lines[name], = ax.plot([], [], [], label=f"{name}-Bahn", color=color, alpha=0.3)
            # Der Planet (Punkt) bekommt die kräftige Farbe
            points[name], = ax.plot([], [], [], 'o', markersize=8, label=name, color=color)

        ax.legend()
        
        # Dynamische Achsen-Skalierung

        # Suche größten Koordinatenwert aller Körper
        max_val = 0
        for pos_array in data_source.values():
            max_val = max(max_val, np.max(np.abs(pos_array)))
        
        # Achsen-Limits setzen (+ 10% Puffer am Rand)
        ax.set_xlim(-max_val * 1.1, max_val * 1.1)
        ax.set_ylim(-max_val * 1.1, max_val * 1.1)
        ax.set_zlim(-max_val * 1.1, max_val * 1.1)
        ax.set_xlabel('X Position')
        ax.set_ylabel('Y Position')
        ax.set_zlabel('Z Position')
        ax.set_title('3D Erde-Mond-Simulation')

        # Diese Update-Funktion wird für jeden Frame der Animation aufgerufen
        def update(frame):
            for name, pos_array in data_source.items():
                pos_array = np.array(pos_array)
                
                # Historie bis zum aktuellen Frame für die Flugbahn
                x_traj = pos_array[:frame+1, 0]
                y_traj = pos_array[:frame+1, 1]
                z_traj = pos_array[:frame+1, 2]
                
                lines[name].set_data(x_traj, y_traj)
                lines[name].set_3d_properties(z_traj)
                
                # Nur der aktuelle Frame für den Planeten-Punkt
                points[name].set_data([pos_array[frame, 0]], [pos_array[frame, 1]])
                points[name].set_3d_properties([pos_array[frame, 2]])
                
            return list(lines.values()) + list(points.values())

        # Anzahl der Frames aus dem ersten Datensatz ermitteln
        num_frames = len(list(data_source.values())[0])
        
        # Animation generieren
        ani = animation.FuncAnimation(fig, update, frames=num_frames, interval=50, blit=False)
        
        # Figur wird geschlossen, damit Jupyter nicht zusätzlich ein statisches Bild anzeigt
        plt.close(fig) 
        
        # Gibt ein interaktives JavaScript-HTML-Widget zurück
        return HTML(ani.to_jshtml())


In [ ]:
# Setup – Zeiteinheit: Tage | Positionen: Meter | Massen: Kilogramm
simulation = Simulation(1)  # Zeitschritt: 0.1 Tage (≈ 2,4 Stunden)


# Erste Version bei der die Erde nicht stationär ist
erde = Body("Erde", [0, 0, 0], [0, 0, 0], 6371000, 5.972e24, "blue")
mond = Body("Mond", [384400000, 0, 0], [0, 1022 * 86400, 0], 1737000, 7.348e22, "grey")
#                                          ↑ 1022 m/s × 86400 s/Tag → m/Tag

# Zweite Version bei der die Erde zuerst einen Impuls in die entgegengesetzte Richtung bekommt um stationär zu bleiben
#erde = Body("Erde", [0, 0, 0], [0, -12.573 * 86400, 0], 6371000, 5.972e24, "blue")
# mond = Body("Mond", [384400000, 0, 0], [0, 1022 * 86400, 0], 1737000, 7.348e22, "grey")

simulation.add_body(erde)
simulation.add_body(mond)

simulation.run(360)

# Visualizer
vis = Visualizer(simulation)

# 1. Standardansicht (schräg):
# meine_animation = vis.animate_3d()

# 2. Ansicht von oben (Draufsicht auf die X/Y Ebene):
anim = vis.animate_3d()

# 3. Ansicht von der Seite:
# meine_animation = vis.animate_3d(view='side')

anim